# Gene Regulatory Network Inference

**Tier 3 — Applied Bioinformatics | Module 28 · Notebook 3**

*Prerequisites: Notebook 2 (Network Modules), Module 03 (RNA-seq Analysis)*

---

**By the end of this notebook you will be able to:**
1. Distinguish GRN inference from PPI network construction
2. Apply GENIE3 (Random Forest) to infer regulatory edges from expression data
3. Identify network motifs (feed-forward loops, autoregulatory motifs)
4. Validate inferred regulations against known TF binding data (JASPAR/ENCODE)
5. Visualize a subnetwork around a master regulator TF


**Key resources:**
- [GENIE3 documentation](https://bioconductor.org/packages/release/bioc/html/GENIE3.html)
- [ARACNE web server](http://califano.c2b2.columbia.edu/aracne)
- [TRRUST — curated TF-target database](https://www.grnpedia.org/trrust/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Network Modules](./02_network_modules.ipynb) | [Module Overview](../README.md)

---

## 1. Gene Regulatory Networks vs PPI Networks

> Explain the difference: GRNs are directed (TF → target gene), context-specific, and inferred from expression data. PPI networks are undirected (physical interaction). Overview of GRN inference methods: GENIE3, ARACNE, SCENIC.

## 2. GENIE3: Random Forest GRN Inference

> Train GENIE3 on a normalized expression matrix. Explain the feature importance score as a regulatory weight. Output: ranked edge list (TF → target, weight).

In [ ]:
# Example: GENIE3 in R (via rpy2)
# library(GENIE3)
# regulators <- c('TP53', 'MYC', 'E2F1', 'FOXM1', 'NF1')  # TFs of interest
# weight_matrix <- GENIE3(exprMatrix=expr_matrix, regulators=regulators)
# link_list <- getLinkList(weight_matrix, reportMax=10000)
# head(link_list)

## 3. Network Motif Analysis

> Build directed GRN from top edges. Count feed-forward loops (FFL) and autoregulatory motifs using networkx. Compare observed vs expected motif counts (Z-score vs random networks).

In [ ]:
import networkx as nx
import pandas as pd

# Example: Build directed GRN
# grn = nx.from_pandas_edgelist(link_list.head(5000),
#     source='regulatoryGene', target='targetGene',
#     edge_attr='weight', create_using=nx.DiGraph())

# Example: Count feed-forward loops (triads)
# ffl_count = 0
# for a in grn.nodes():
#     for b in grn.successors(a):
#         for c in grn.successors(b):
#             if grn.has_edge(a, c):  # a->b->c and a->c = coherent FFL
#                 ffl_count += 1
# print(f'Feed-forward loops: {ffl_count}')

## 4. Validation Against Known TF Targets

> Load TRRUST curated TF-target pairs. Compute precision and recall of GENIE3 predictions against known interactions. PR curve.

In [ ]:
# Example: Load TRRUST and evaluate predictions
# trrust = pd.read_csv('trrust_rawdata.human.tsv', sep='\t',
#                      names=['TF', 'target', 'relationship', 'ref'])
# known_edges = set(zip(trrust['TF'], trrust['target']))
# predicted_edges = set(zip(link_list['regulatoryGene'], link_list['targetGene']))
# tp = len(known_edges & predicted_edges)
# precision = tp / len(predicted_edges)
# recall = tp / len(known_edges)
# print(f'Precision: {precision:.3f}, Recall: {recall:.3f}')

## 5. Master Regulator Subnetwork

> Extract the 1-hop ego network around TP53 from the inferred GRN. Visualize with directed edges. Annotate target genes by their known function (tumor suppressor, oncogene).

In [ ]:
import matplotlib.pyplot as plt

# Example: TP53 subnetwork
# tp53_targets = list(grn.successors('TP53'))
# ego = grn.subgraph(['TP53'] + tp53_targets)

# pos = nx.shell_layout(ego)
# plt.figure(figsize=(12, 12))
# nx.draw(ego, pos, with_labels=True, node_color='lightcoral',
#         arrows=True, arrowsize=20, node_size=500, font_size=9)
# plt.title('TP53 Regulatory Network (GENIE3 inferred)')
# plt.show()

## Summary

> Recap GRN inference and validation. Discuss limitations (correlation ≠ causation, sample size requirements). Mention SCENIC for single-cell GRN inference as an extension.

---

[← Previous: Network Modules](./02_network_modules.ipynb) | [Module Overview](../README.md)

---